# Caption smoke test -- Qwen2.5-VL-7B-Instruct on surgical frames

Swapped model, not the S3 experiment's stack: **Qwen2.5-VL-7B-Instruct**
instead of Qwen3.5-9B+adapter+CountGD. Reasons: fully open on Hugging Face
(no S3 credentials, no adapter, no CountGD/GroundingDINO CUDA build), and
small enough to be safe on whatever Colab GPU tier you have. This is a
qualitative check only -- generate a caption, look at it yourself, no
scoring/evaluation harness.

If you hit an OOM: drop `MODEL_ID` to `Qwen/Qwen2.5-VL-3B-Instruct` (same
interface, ~half the memory), or flip `LOAD_IN_4BIT = True` below.

## 0. Setup

In [ ]:

import sys, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/endofm-benchmark")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.49.0", "accelerate", "qwen-vl-utils", "bitsandbytes",
                    "opencv-python-headless", "pillow"], check=True)
else:
    PROJECT_ROOT = Path(r"E:/endofm-benchmark")

sys.path.insert(0, str(PROJECT_ROOT))
from frame_utils import extract_frame

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 1. Load model

In [ ]:

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"   # drop to "Qwen/Qwen2.5-VL-3B-Instruct" if this OOMs
LOAD_IN_4BIT = False                        # flip to True for a smaller memory footprint

load_kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    load_kwargs = dict(load_in_4bit=True, device_map="auto")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Loaded:", MODEL_ID)


## 2. Pick a frame and look at it

`VIDEO_PATH`/`TIMESTAMP_S` are the two things to change to try different
frames. Defaults to the PitVis smoke-test video already on Drive.

In [ ]:

VIDEO_PATH = PROJECT_ROOT / "pitvis" / "video_01.mp4"
TIMESTAMP_S = 331   # seconds into the video

image = extract_frame(VIDEO_PATH, TIMESTAMP_S)
display(image)


## 3. Generate a caption

In [ ]:

PROMPT = "Describe what is happening in this surgical video frame in detail."

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ],
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256)

caption = processor.batch_decode(
    output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
)[0]
print(caption)


## 4. Try several frames at once (optional)

Same pattern, looped over a few timestamps so you can eyeball a batch
without re-running cells one at a time.

In [ ]:

def caption_frame(image, prompt=PROMPT):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)
    return processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

TIMESTAMPS_TO_TRY = [60, 604, 1800, 3600]

for t in TIMESTAMPS_TO_TRY:
    frame = extract_frame(VIDEO_PATH, t)
    display(frame)
    print(f"t={t}s:", caption_frame(frame))
    print("-" * 80)
